# Aula 7: demonstração de NumPy, pandas, Matplotlib e Seaborn

Material de apoio da Aula 7 (Análise de Dados com Python). Cobre as quatro
bibliotecas apresentadas na aula, na mesma ordem dos slides, com código
rodando de verdade em cima da base de fraude em cartão da Aula 6. Sem
preocupação de tempo de apresentação: é para ler, rodar e revisitar com
calma, não para caber numa demonstração cronometrada.

**Antes de começar:** os arquivos CSV desta demonstração são os mesmos da
Aula 6. Se este notebook estiver na mesma pasta do Drive que `csv/`, não
precisa mudar nada. Se estiver em outro lugar, ajuste `PASTA_DADOS` na célula
abaixo.

## 0. Preparação

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PASTA_DADOS = "/content/drive/MyDrive/AI Talent Academy/Aula 06/dados/csv"

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
categorias = pd.read_csv(f"{PASTA_DADOS}/categorias.csv")
clientes = pd.read_csv(f"{PASTA_DADOS}/clientes.csv")
lojistas = pd.read_csv(f"{PASTA_DADOS}/lojistas.csv")
cartoes = pd.read_csv(f"{PASTA_DADOS}/cartoes.csv")
transacoes = pd.read_csv(f"{PASTA_DADOS}/transacoes.csv")
contestacoes = pd.read_csv(f"{PASTA_DADOS}/contestacoes.csv")
transacoes["data_hora"] = pd.to_datetime(transacoes["data_hora"])

print("transacoes:", transacoes.shape)
print("clientes:", clientes.shape)

In [ ]:
transacoes.head()

## 1. NumPy: a base por trás do pandas

Por baixo de qualquer coluna numérica do pandas existe um array do NumPy.
`to_numpy()` extrai esse array puro, sem rótulo de linha, sem nome de
coluna, só os números.

In [ ]:
import numpy as np

valores = transacoes["valor"].to_numpy()
type(valores)

In [ ]:
valores[:5]

### 1.1 Funções estatísticas prontas

As mesmas contas que o `describe()` do pandas mostra, agora chamadas direto
no array.

In [ ]:
print("média:", valores.mean())
print("soma:", valores.sum())
print("mínimo:", valores.min())
print("máximo:", valores.max())
print("desvio padrão:", valores.std())

### 1.2 Operação vetorizada

A conta se aplica em todos os elementos de uma vez, sem laço.

In [ ]:
valores_com_taxa = valores * 1.03
valores_com_taxa[:5]

### 1.3 Indexação booleana

A mesma lógica de filtro que o pandas usa por baixo. `valores > 1000` gera
um array de `True`/`False`, e o array original devolve só as posições
marcadas como `True`.

In [ ]:
acima_de_mil = valores[valores > 1000]
print("quantas transações acima de R$1.000:", len(acima_de_mil))

### 1.4 Um array com mais de uma dimensão

O array de uma coluna do pandas é sempre unidimensional, mas o NumPy
também guarda tabela e imagem como array de duas ou mais dimensões.

In [ ]:
matriz = np.array([[1, 2, 3], [4, 5, 6]])
print("formato da matriz:", matriz.shape)
matriz

## 2. pandas: filtrar, agrupar e juntar

As mesmas 6 tabelas da Aula 6, agora manipuladas direto em pandas, sem
passar por banco nenhum.

### 2.1 O índice

Por padrão, o índice é só um número sequencial. `set_index` troca por uma
coluna que faça mais sentido, e `reset_index` desfaz essa troca.

In [ ]:
transacoes.index

In [ ]:
transacoes_por_id = transacoes.set_index("id_transacao")
transacoes_por_id.head(3)

### 2.2 Tipos de dado

`data_hora` já foi convertida na preparação, lá em cima. Sem essa
conversão, ela chegaria como texto simples, sem nenhuma operação de data
liberada.

In [ ]:
transacoes.dtypes

### 2.3 Um primeiro raio-x da tabela

In [ ]:
transacoes.info()

In [ ]:
transacoes.describe()

### 2.4 Contando valores únicos

In [ ]:
transacoes["canal"].value_counts()

In [ ]:
transacoes["id_lojista"].nunique()

### 2.5 Filtrando linhas, a versão pandas do WHERE

In [ ]:
caras = transacoes[transacoes["valor"] > 1000]
len(caras)

### 2.6 loc e iloc: duas formas de selecionar

Com o índice trocado para `id_transacao`, `loc` e `iloc` deixam de
coincidir. `loc[500]` busca o RÓTULO 500. `iloc[500]` busca a POSIÇÃO 500,
que é outra linha.

In [ ]:
transacoes_por_id.loc[500]

In [ ]:
transacoes_por_id.iloc[500]

### 2.7 Valores ausentes

No pandas, valor ausente se chama NaN, o mesmo papel do NULL em SQL.

In [ ]:
clientes.isna().sum()

In [ ]:
clientes["telefone"] = clientes["telefone"].fillna("não informado")
clientes["telefone"].isna().sum()

### 2.8 Criando uma coluna nova

In [ ]:
transacoes["valor_com_taxa"] = transacoes["valor"] * 1.03
transacoes[["valor", "valor_com_taxa"]].head()

### 2.9 Agrupando e somando, a versão pandas do GROUP BY

In [ ]:
resumo = transacoes.groupby("id_lojista")["valor"].sum()
resumo.sort_values(ascending=False).head(10)

### 2.10 Mais de uma agregação de uma vez

In [ ]:
transacoes.groupby("id_lojista")["valor"].agg(["count", "sum", "mean"]).head(10)

### 2.11 Tabela dinâmica: pivot_table

Quanto cada lojista vendeu, separado por canal.

In [ ]:
pd.pivot_table(transacoes, index="id_lojista", columns="canal",
               values="valor", aggfunc="sum").head(10)

### 2.12 Juntando tabelas, a versão pandas do JOIN

Do jeito real: transação vem de cartão, cartão vem de cliente. São dois
merge em sequência, não um só.

In [ ]:
completo = (
    transacoes
    .merge(cartoes, on="id_cartao")
    .merge(clientes, on="id_cliente", suffixes=("_cartao", "_cliente"))
    .merge(lojistas, on="id_lojista", suffixes=("_cliente", "_lojista"))
)
len(completo)

In [ ]:
completo[["id_transacao", "nome_cliente", "nome_lojista", "valor"]].head(10)

## 3. Matplotlib: transformando tabela em gráfico

A base de todo gráfico feito em Python, inclusive os do Seaborn logo
abaixo.

### 3.1 Gráfico de barras

O `resumo` do `groupby`, agora virando desenho.

In [ ]:
import matplotlib.pyplot as plt

top10 = resumo.sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 5))
plt.bar(top10.index.astype(str), top10.values)
plt.title("Os 10 lojistas com maior valor total")
plt.xlabel("id_lojista")
plt.ylabel("Valor total (R$)")
plt.xticks(rotation=45)
plt.show()

### 3.2 Gráfico de linha, evolução no tempo

Agrupando por mês, dá para ver a tendência do valor movimentado ao longo
do período da base.

In [ ]:
transacoes["mes"] = transacoes["data_hora"].dt.to_period("M")
por_mes = transacoes.groupby("mes")["valor"].sum()

plt.figure(figsize=(10, 4))
plt.plot(por_mes.index.astype(str), por_mes.values, marker="o")
plt.title("Valor total movimentado por mês")
plt.xlabel("Mês")
plt.ylabel("Valor total (R$)")
plt.xticks(rotation=45)
plt.show()

### 3.3 Histograma, direto no Matplotlib

Sem o Seaborn por cima, é preciso configurar mais coisa à mão: número de
faixas, título, tudo manual.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(transacoes["valor"], bins=30)
plt.title("Distribuição do valor das transações")
plt.xlabel("Valor (R$)")
plt.ylabel("Quantidade de transações")
plt.show()

## 4. Seaborn: bonito por padrão, e entende DataFrame direto

Mesmo Matplotlib por baixo, com bem menos configuração manual.

### 4.1 Histograma, agora pelo Seaborn

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(transacoes["valor"])
plt.title("Distribuição do valor das transações")
plt.show()

### 4.2 Boxplot, comparando distribuição por categoria

Compara o valor das transações presenciais contra as online, numa figura
só. Um gráfico assim, no Matplotlib puro, levaria bem mais código.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=transacoes, x="canal", y="valor")
plt.title("Distribuição do valor por canal")
plt.show()

### 4.3 Barras com agregação embutida

`sns.barplot` já calcula a média por categoria sozinho, sem precisar de um
`groupby` antes.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=transacoes, x="status", y="valor")
plt.title("Valor médio por status da transação")
plt.show()

## Pronto

Este notebook cobre as quatro bibliotecas apresentadas na Aula 7, rodando
de verdade em cima da mesma base de fraude em cartão que a turma já
conhece. Na Aula 8, vocês vão fazer o mesmo percurso, agora escrevendo o
código vocês mesmos.